# 36 · Motivo da pesquisa de saída × tipo de treino

Cruza o motivo declarado na pesquisa de saída (`motivo_ajustado`, que já lê o comentário e cria
`objetivo_cumprido`, ver decisoes.md) com o tipo de treino principal de cada respondente.

- **Quem responde**: só quem cancelou pelo site, de jun a set/2026 (641 contas; 10% das saídas do
  período). Não generalizar para todas as saídas.
- **Tipo principal**: o mais frequente nos 90 dias antes da resposta (empate: mais minutos).
- **Engajamento**: nível de `src/engajamento.py` na véspera da resposta.
- **Ativos / churn**: situação da assinatura do respondente no corte (30/09/2026).

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
TIPOS = ["corrida", "funcional", "caminhada", "forca", "pedal", "yoga", "sem treino"]

pesq = pd.read_parquet(P / "pesquisa_saida_limpa.parquet")
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "canal_cobranca", "inicio", "data_fim", "motivo_fim", "ativa_no_corte", "renovacao_automatica"])
usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sistema"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "duracao_min"])
print("respostas:", len(pesq), "| contas:", pesq.user_id.nunique())
r = pesq[pesq.resposta_mais_recente].merge(assin, on="user_id", how="left", validate="one_to_one").merge(usuarios, on="user_id", how="left", validate="one_to_one")
print("resposta mais recente por conta:", len(r), "| sem assinatura:", int(r.plano_atual.isna().sum()))
print("período das respostas:", r.data_resposta.min().date(), "a", r.data_resposta.max().date())
print("plano:", r.plano_atual.value_counts().to_dict(), "| canal:", r.canal_cobranca.value_counts().to_dict(), "| sistema:", r.sistema.value_counts().to_dict())

respostas: 643 | contas: 641
resposta mais recente por conta: 641 | sem assinatura: 0
período das respostas: 2026-06-01 a 2026-09-30
plano: {'mensal': 334, 'anual': 307} | canal: {'web_cartao': 436, 'web_pix': 205} | sistema: {'android': 407, 'ios': 234}


## 1. Tipo de treino e nível de engajamento de cada respondente

In [2]:
a = ativ[ativ.user_id.isin(r.user_id)].merge(r[["user_id", "data_resposta"]], on="user_id")
a = a[(a.data < a.data_resposta) & (a.data >= a.data_resposta - pd.Timedelta(days=90))]
g = a.groupby(["user_id", "tipo"]).agg(n=("data", "size"), minutos=("duracao_min", "sum")).reset_index().sort_values(["user_id", "n", "minutos"], ascending=[True, False, False])
top = g.drop_duplicates("user_id").set_index("user_id")
r = r.join(pd.DataFrame({"principal": top.tipo, "treinos_90d": g.groupby("user_id").n.sum(), "n_tipos": g.groupby("user_id").tipo.nunique(), "peso_principal": top.n / g.groupby("user_id").n.sum()}), on="user_id")
print("respondentes com treino nos 90 dias antes da resposta:", int(r.principal.notna().sum()), "| sem treino:", int(r.principal.isna().sum()))
r["principal"] = r.principal.fillna("sem treino"); r["n_tipos"] = r.n_tipos.fillna(0).astype(int); r["treinos_90d"] = r.treinos_90d.fillna(0)
e = niveis_engajamento(ativ, r.user_id, r.set_index("user_id").data_resposta - pd.Timedelta(days=1), inicio=r.set_index("user_id").inicio)
r = r.merge(e[["user_id", "nivel", "treinos_30d", "dias_sem_treinar"]], on="user_id", validate="one_to_one")
print("depois das junções:", len(r))
r["ambiente"] = np.where(r.principal.isin(["corrida", "caminhada", "pedal"]), "ar livre", np.where(r.principal == "sem treino", "sem treino", "indoor"))
r["variedade"] = pd.cut(r.n_tipos, [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
MOT = r.motivo_ajustado.value_counts().index.tolist()
print(); print("motivo (ajustado pelo comentário)"); x = r.motivo_ajustado.value_counts().rename("respostas").to_frame(); x["%"] = x.respostas / x.respostas.sum() * 100; print(x)
print(); print("tipo principal dos respondentes"); x = r.principal.value_counts().rename("respostas").to_frame(); x["%"] = x.respostas / x.respostas.sum() * 100; print(x)

respondentes com treino nos 90 dias antes da resposta: 611 | sem treino: 30
depois das junções: 641

motivo (ajustado pelo comentário)
                   respostas    %
motivo_ajustado                  
preco                    199 31.0
objetivo_cumprido        156 24.3
outro                    138 21.5
nao_uso                   76 11.9
outro_app                 31  4.8
problema_tecnico          27  4.2
saude                     14  2.2

tipo principal dos respondentes
            respostas    %
principal                 
corrida           302 47.1
funcional         117 18.3
caminhada          65 10.1
pedal              60  9.4
forca              55  8.6
sem treino         30  4.7
yoga               12  1.9


## 2. Motivo × tipo de treino principal

In [3]:
n = pd.crosstab(r.principal, r.motivo_ajustado, margins=True, margins_name="total").reindex(index=[t for t in TIPOS if t in r.principal.unique()] + ["total"], columns=MOT + ["total"])
print("== respostas (contas)"); print(n); print()
pl = (pd.crosstab(r.principal, r.motivo_ajustado, normalize="index", margins=True, margins_name="total") * 100).reindex(index=n.index, columns=MOT)
print("== % da linha: de quem tem cada tipo principal, que motivo deu"); print(pl); print()
pc = (pd.crosstab(r.principal, r.motivo_ajustado, normalize="columns", margins=True, margins_name="total") * 100).reindex(index=n.index[:-1], columns=MOT + ["total"])
print("== % da coluna: de quem deu cada motivo, que tipo treinava"); print(pc); print()
print("== por ambiente (% da linha)"); print((pd.crosstab(r.ambiente, r.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(r.ambiente.value_counts().rename("respostas")))
n.to_csv(S / "pesquisa_saida_motivo_por_tipo_treino.csv", encoding="utf-8-sig")

== respostas (contas)
motivo_ajustado  preco  objetivo_cumprido  outro  nao_uso  outro_app  problema_tecnico  saude  total
principal                                                                                           
corrida             83                100     56       29         13                13      8    302
funcional           36                 19     40       12          5                 1      4    117
caminhada           23                 10     12       10          2                 8      0     65
forca               19                 10     13        5          5                 1      2     55
pedal               18                 16     12        8          2                 4      0     60
yoga                 5                  1      3        2          1                 0      0     12
sem treino          15                  0      2       10          3                 0      0     30
total              199                156    138       76         31 

In [4]:
print("== por variedade de tipos (% da linha)"); print((pd.crosstab(r.variedade, r.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(r.variedade.value_counts().rename("respostas")))
print()
print("== por sistema (% da linha)"); print((pd.crosstab(r.sistema, r.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(r.sistema.value_counts().rename("respostas")))
print()
k = r[r.sistema == "ios"]
print("== só iOS, motivo × ambiente (% da linha):", len(k), "respostas"); print((pd.crosstab(k.ambiente, k.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(k.ambiente.value_counts().rename("respostas")))
print()
print("== corrida contra os demais tipos (% da linha)")
r["corrida"] = np.where(r.principal == "corrida", "corrida", np.where(r.principal == "sem treino", "sem treino", "outros tipos"))
print((pd.crosstab(r.corrida, r.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(r.corrida.value_counts().rename("respostas")))
print(pd.crosstab(r.corrida, r.motivo_ajustado).reindex(columns=MOT))
print()
print("== por plano (% da linha)"); print((pd.crosstab(r.plano_atual, r.motivo_ajustado, normalize="index") * 100).reindex(columns=MOT).join(r.plano_atual.value_counts().rename("respostas")))

== por variedade de tipos (% da linha)
            preco  objetivo_cumprido  outro  nao_uso  outro_app  problema_tecnico  saude  respostas
variedade                                                                                          
sem treino   50.0                0.0    6.7     33.3       10.0               0.0    0.0         30
1 tipo       34.8                8.7   17.4     21.7        0.0              13.0    4.3         23
2 tipos      37.5               12.5   10.4     18.8       12.5               6.2    2.1         48
3 tipos      42.0               17.4   17.4     13.0        1.4               5.8    2.9         69
4+ tipos     27.4               28.9   24.4      9.1        4.5               3.6    2.1        471

== por sistema (% da linha)
         preco  objetivo_cumprido  outro  nao_uso  outro_app  problema_tecnico  saude  respostas
sistema                                                                                         
android   31.4               22.4   21

## 3. Engajamento na véspera da resposta, por motivo

In [5]:
n = pd.crosstab(r.motivo_ajustado, r.nivel).reindex(index=MOT, columns=NIVEIS, fill_value=0)
t = (n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("respostas"))
t["treinos 30d (média)"] = r.groupby("motivo_ajustado").treinos_30d.mean()
t["treinos 90d (média)"] = r.groupby("motivo_ajustado").treinos_90d.mean()
t["nº de tipos (média)"] = r.groupby("motivo_ajustado").n_tipos.mean()
t["peso do principal % (média)"] = r.groupby("motivo_ajustado").peso_principal.mean() * 100
t["meses de casa (mediana)"] = r.assign(m=(r.data_resposta - r.inicio).dt.days / 30.44).groupby("motivo_ajustado").m.median()
print(t); print()
n = pd.crosstab(r.principal, r.nivel).reindex(columns=NIVEIS, fill_value=0)
print("nível na véspera por tipo principal (% da linha)"); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("respostas")))

                   1. alto  2. médio  3. baixo  4. possível churn  respostas  treinos 30d (média)  treinos 90d (média)  nº de tipos (média)  peso do principal % (média)  meses de casa (mediana)
motivo_ajustado                                                                                                                                                                                  
preco                 51.8      32.2       3.0               13.1        199                  7.3                 23.1                  3.5                         63.3                      5.4
objetivo_cumprido     59.6      37.2       2.6                0.6        156                  9.5                 35.4                  4.2                         67.9                      5.4
outro                 48.6      44.9       3.6                2.9        138                  7.6                 29.3                  4.0                         59.7                      7.2
nao_uso               46.1    

## 4. Ativos e churn: situação dos respondentes no corte

In [6]:
r["situacao"] = np.where(r.ativa_no_corte & (r.renovacao_automatica == "nao"), "ativo, renovação desligada", np.where(r.ativa_no_corte, "ativo, renovação ligada", "saiu até o corte"))
print(r.situacao.value_counts().to_dict())
print("motivo_fim de quem saiu:", r[~r.ativa_no_corte].motivo_fim.value_counts().to_dict())
r["dias_ate_fim"] = (r.data_fim - r.data_resposta).dt.days
print("dias da resposta ao fim da assinatura (quem saiu):", r.dias_ate_fim.describe()[["count", "min", "25%", "50%", "75%", "max"]].round(0).to_dict())
print(); print("situação no corte por motivo (% da linha)")
print((pd.crosstab(r.motivo_ajustado, r.situacao, normalize="index") * 100).reindex(MOT).join(r.motivo_ajustado.value_counts().rename("respostas")))
print(); print("situação no corte por tipo principal (% da linha)")
print((pd.crosstab(r.principal, r.situacao, normalize="index") * 100).join(r.principal.value_counts().rename("respostas")))
print()
# treino depois da resposta: quem disse objetivo cumprido parou?
d = ativ[ativ.user_id.isin(r.user_id)].merge(r[["user_id", "data_resposta"]], on="user_id")
pos = d[(d.data > d.data_resposta) & (d.data <= d.data_resposta + pd.Timedelta(days=30))].groupby("user_id").size().rename("treinos_30d_depois")
rr = r[r.data_resposta <= CORTE - pd.Timedelta(days=30)].join(pos, on="user_id").fillna({"treinos_30d_depois": 0})
print("respondentes com 30 dias de observação depois da resposta:", len(rr))
print(rr.groupby("motivo_ajustado").agg(respostas=("user_id", "size"), treinos_30d_antes=("treinos_30d", "mean"), treinos_30d_depois=("treinos_30d_depois", "mean"), pct_treinou_depois=("treinos_30d_depois", lambda s: (s > 0).mean() * 100)).reindex(MOT))

{'saiu até o corte': 353, 'ativo, renovação desligada': 263, 'ativo, renovação ligada': 25}
motivo_fim de quem saiu: {'renovacao_desligada': 352, 'falha_pagamento': 1}
dias da resposta ao fim da assinatura (quem saiu): {'count': 353.0, 'min': 0.0, '25%': 9.0, '50%': 17.0, '75%': 24.0, 'max': 85.0}

situação no corte por motivo (% da linha)
                   ativo, renovação desligada  ativo, renovação ligada  saiu até o corte  respostas
motivo_ajustado                                                                                    
preco                                    40.2                      1.0              58.8        199
objetivo_cumprido                        39.1                      4.5              56.4        156
outro                                    45.7                      4.3              50.0        138
nao_uso                                  40.8                      6.6              52.6         76
outro_app                                41.9             

In [7]:
print("comentários mais comuns por motivo ajustado (até 4 por motivo)")
for m in MOT:
    c = r[(r.motivo_ajustado == m) & r.comentario_norm.notna()].comentario_norm.value_counts().head(4)
    print(f"-- {m} ({int((r.motivo_ajustado == m).sum())} respostas, {int(r[r.motivo_ajustado == m].comentario_norm.notna().sum())} com comentário)")
    for txt, q in c.items(): print(f"   {q:>3}  {txt[:90]}")

comentários mais comuns por motivo ajustado (até 4 por motivo)
-- preco (199 respostas, 122 com comentário)
    11  app bugado depois da atualização
    10  vou dar um tempo
     9  o gps não funciona no meu celular
     7  sei lá, perdi a vontade
-- objetivo_cumprido (156 respostas, 156 com comentário)
    28  treinei pra prova e consegui, objetivo cumprido
    27  já fiz a prova, agora não preciso mais
    25  era só até a prova
     6  treinei pra apresentação pros investidores e consegui, objetivo cumprido
-- outro (138 respostas, 78 com comentário)
     6  trabalho novo, não dá
     4  vou passar uns meses fora
     4  mudei de vida, sem tempo
     3  já fiz a viagem pro nordeste, agora não preciso mais
-- nao_uso (76 respostas, 46 com comentário)
     9  não estou usando
     8  vou dar um tempo
     7  sei lá, perdi a vontade
     7  virou rotina chata
-- outro_app (31 respostas, 21 com comentário)
     4  vou dar um tempo
     2  o gps não funciona no meu celular
     2  parado